# MLP Fusion Classifier — Riot vs. Normal

Trains an MLP on the fused motion+visual feature vectors from
`feature_extraction_fusion.ipynb` (any `features_*.csv` under
`My Drive/CVPHG/fusion_features/` -- merged the same way
`ablation_fusion_branches.ipynb` does, so this works whether you ran
extraction as one combined pass or as separate branch-selective runs):
LK optical flow summary stats + RF-DETR detection counts + DINOv2 +
VideoMAE embeddings, concatenated per clip.

**Binary riot/normal only** -- the "Protest" class from the original
3-way architecture diagram has no data source yet (confirmed: the
Kaggle `jpmiller/protests-against-police-violence` dataset is 744/748
Capitol riot footage, no peaceful-protest content). This trains and
reports what's actually supported by the data in hand.

**Class imbalance (changed 2026-09-26):** if `xdviolence_normal_pull.ipynb`
was run with the full Normal class (2346 clips) against the 485-clip riot
class, this is a ~4.8:1 imbalanced problem, not the balanced 1:1 set the
original 92.27% result was measured on. Raw accuracy alone is misleading
on imbalanced data (predicting "normal" for everything would already
score ~83%), so this notebook reports **balanced accuracy** (average of
per-class recall) alongside raw accuracy -- always read both together.

**MLP first, per current plan** -- a Transformer fusion head is a
separate later notebook, after these MLP results are reviewed.

## Step 0 — Setup

In [ ]:
!pip install scikit-learn -q

In [ ]:
import os
import csv
import glob
import json

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                              precision_recall_fscore_support,
                              confusion_matrix, classification_report)
from google.colab import drive

drive.mount('/content/drive')

FEATURES_DIR = '/content/drive/MyDrive/CVPHG/fusion_features'
RESULTS_DIR = os.path.join(FEATURES_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42

## Step 1 — Load features and prepare train/test split

In [ ]:
csv_paths = sorted(glob.glob(os.path.join(FEATURES_DIR, 'features_*.csv')))
print(f"Found {len(csv_paths)} feature file(s):")
for p in csv_paths:
    print(' ', os.path.basename(p))
if not csv_paths:
    raise SystemExit(
        'No features_*.csv found -- run feature_extraction_fusion.ipynb '
        'with at least one BRANCHES selection first.'
    )

dfs = [pd.read_csv(p) for p in csv_paths]
df = dfs[0]
for other in dfs[1:]:
    new_cols = [c for c in other.columns if c not in df.columns]
    df = df.merge(other[['video_id'] + new_cols], on='video_id', how='outer')
df['label'] = df['label'].fillna(method='ffill').fillna(method='bfill')
df = df.dropna()  # a clip missing from one of the merged files can't be used until that file also covers it

print(f"\n{len(df)} clips loaded: {df['label'].value_counts().to_dict()}")

feature_cols = [c for c in df.columns if c not in ('video_id', 'label', 'split')]
print(f"{len(feature_cols)} feature dimensions")

X = df[feature_cols].values
y = (df['label'] == 'riot').astype(int).values  # riot=1, normal=0

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, df['video_id'].values, test_size=0.2, random_state=SEED, stratify=y,
)
print(f"Train: {len(X_train)} ({y_train.sum()} riot / {len(y_train) - y_train.sum()} normal)")
print(f"Test:  {len(X_test)} ({y_test.sum()} riot / {len(y_test) - y_test.sum()} normal)")

## Step 2 — Scale features and train the MLP

Standardization matters here: DINOv2/VideoMAE embedding dims and raw
LK/RF-DETR scalars (e.g. pixel-magnitude flow, person counts) are on
very different scales -- an unscaled MLP would be dominated by
whichever branch happens to have larger raw magnitudes, not whichever
is actually more informative.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 64),
    activation='relu',
    alpha=1e-4,
    early_stopping=True,
    n_iter_no_change=15,
    max_iter=500,
    random_state=SEED,
)
mlp.fit(X_train_scaled, y_train)
print(f"Training stopped after {mlp.n_iter_} iterations (best val score: {mlp.best_validation_score_:.4f})")

## Step 3 — Evaluate on the held-out test set

In [ ]:
y_pred = mlp.predict(X_test_scaled)
y_proba = mlp.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=[0, 1], zero_division=0,
)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

print(f"Accuracy: {accuracy:.2%}  |  Balanced accuracy: {balanced_accuracy:.2%}\n")
print(classification_report(y_test, y_pred, target_names=['normal', 'riot'], zero_division=0))
print("Confusion matrix (rows=true, cols=pred) [normal, riot]:")
print(cm)

## Step 4 — Save results, matching this project's reporting convention

Same shape as `results/*.md` + `*_predictions.csv` pairs elsewhere in
this project (see `scripts/evaluate.py`), so this slots into
`docs/findings.md` the same way every prior result has.

In [ ]:
predictions_df = pd.DataFrame({
    'video_id': id_test,
    'true_label': ['riot' if v == 1 else 'normal' for v in y_test],
    'predicted_label': ['riot' if v == 1 else 'normal' for v in y_pred],
    'confidence': np.where(y_pred == 1, y_proba, 1 - y_proba),
})
predictions_csv = os.path.join(RESULTS_DIR, 'mlp_fusion_riot_normal_predictions.csv')
predictions_df.to_csv(predictions_csv, index=False)
print(f"Saved predictions -> {predictions_csv}")

In [ ]:
classes = ['normal', 'riot']
lines = []
lines.append('# MLP Fusion Classifier Results (Riot vs. Normal)\n')
lines.append(
    'Supervised MLP (256,64 hidden units) trained on fused features: '
    'Lucas-Kanade optical flow summary stats + RF-DETR detection counts + '
    'DINOv2 frame embeddings + VideoMAE clip embeddings. '
    'Binary riot/normal only -- see notebook header for why.\n'
)
lines.append(f'**Train clips:** {len(X_train)}  \n**Test clips:** {len(X_test)}  \n**Test accuracy:** {accuracy:.2%}  \n**Balanced accuracy:** {balanced_accuracy:.2%}\n')

lines.append('## Per-class precision / recall / F1\n')
lines.append('| class | precision | recall | f1 | support |')
lines.append('|---|---|---|---|---|')
for i, c in enumerate(classes):
    lines.append(f'| {c} | {precision[i]:.2f} | {recall[i]:.2f} | {f1[i]:.2f} | {support[i]} |')
lines.append('')

lines.append('## Confusion matrix (rows = true label, cols = predicted)\n')
lines.append('| true \\ pred | ' + ' | '.join(classes) + ' |')
lines.append('|---|' + '|'.join(['---'] * len(classes)) + '|')
for i, c in enumerate(classes):
    lines.append(f'| {c} | ' + ' | '.join(str(x) for x in cm[i]) + ' |')
lines.append('')

report_path = os.path.join(RESULTS_DIR, 'mlp_fusion_riot_normal.md')
with open(report_path, 'w') as fh:
    fh.write('\n'.join(lines))
print(f"Saved report -> {report_path}")

In [ ]:
model_path = os.path.join(RESULTS_DIR, 'mlp_fusion_model.joblib')
joblib.dump({'model': mlp, 'scaler': scaler, 'feature_cols': feature_cols}, model_path)
print(f"Saved trained model + scaler -> {model_path}")

## Next steps

- Copy `mlp_fusion_riot_normal.md` + `_predictions.csv` into this
  project's `results/` and log a `docs/findings.md` entry once these
  numbers have actually been reviewed -- tag `[REAL RESULT]` since this
  is real XD-Violence data, not a stand-in, but note the binary-only
  scope (no Protest class) explicitly in the entry, same as every
  other caveat in that log.
- Transformer fusion head is next, per plan -- same features.csv,
  swap the MLPClassifier step for a small Transformer encoder over the
  (unfused) per-branch feature groups, once these MLP results are in.
- If accuracy is suspiciously high/low, check class balance drift
  between train/test (`stratify=y` should prevent this, but confirm)
  and whether the LK motion branch actually varies across clips (some
  riot footage in this dataset is closely cropped, near-static shots --
  worth a manual check before trusting the motion branch's signal).